# stPainter Tutorial

End-to-end walkthrough of using a pretrained **stPainter** model to impute gene expression on a spatial transcriptomics dataset.

**Pipeline**
1. Load the pretrained **VAE** (gene-space ↔ latent compression) and the **Diffusion** model (denoiser in latent space).
2. Load a spatial transcriptomics dataset (`STDataset`).
3. Run `DiffusionModule.impute` to obtain (a) the imputed gene matrix and (b) the enhanced latent embedding.
4. Inspect the results — marker gene before/after imputation, and a UMAP on the latent space.

**Prerequisites**
- `stpainter` conda env (see `requirements.txt`).
- Pretrained checkpoints in `./checkpoint/` (`vae_50.ckpt`, `diffusion_50.ckpt`).
- Demo data at `./data/processed/st_COAD_demo.h5ad`.

This notebook lives at the repo root; launch Jupyter from the repo root so the relative paths below resolve correctly.

## 1. Setup

Make the repo root importable so `src.*` resolves, then pick a device (GPU if available, otherwise CPU).

In [ ]:
import os
import sys
from types import SimpleNamespace

REPO_ROOT = os.path.abspath('.')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import pytorch_lightning as pl
from torch.utils.data import DataLoader

pl.seed_everything(42, workers=True)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Configuration

`DiffusionModule` and `VAEModule` were originally instantiated from `argparse` args. Here we build the same fields with a `SimpleNamespace` — equivalent, just inline.

Toggle between the 50- and 100-dim models by changing `latent_size` (and the matching checkpoint paths).

In [ ]:
LATENT_SIZE = 50  # or 100

args = SimpleNamespace(
    # --- model architecture ---
    input_size=10000,
    latent_size=LATENT_SIZE,
    num_classes=21,
    hidden_size_vae=256,
    num_layer=4,
    patch_size=1,
    hidden_size_sit=128,
    depth=8,
    num_heads=8,
    mlp_ratio=4.0,
    class_dropout_prob=0.1,
    learn_sigma=False,

    # --- inference ---
    batch_size=512,
    num_cpus=4,
    t_forward=0.9,
    cancer_type='COAD',
    gene_sparsity_ratio_file_path='./data/processed/gene_sparsity_ratio.csv',

    # --- transport / sampler ---
    path_type='Linear',
    prediction='velocity',
    loss_weight=None,
    train_eps=None,
    sample_eps=None,
    mode='ODE',
    cfg_scale=1.0,
    num_steps=30,
    sampling_method='euler',
    atol=1e-6,
    rtol=1e-3,
    reverse=False,
    likelihood=False,
)

VAE_CKPT = f'./checkpoint/vae_{LATENT_SIZE}.ckpt'
DIFF_CKPT = f'./checkpoint/diffusion_{LATENT_SIZE}.ckpt'
ST_PATH = './data/processed/st_COAD.h5ad'

assert os.path.exists(VAE_CKPT), VAE_CKPT
assert os.path.exists(DIFF_CKPT), DIFF_CKPT
assert os.path.exists(ST_PATH), ST_PATH
print('All inputs present.')

## 3. Load Pretrained Models

Two stages:
- **VAE** (`scvi.module.VAE`) — encodes 10000-dim gene expression into a compact latent (here, 50-dim).
- **GiT** — Gene Diffusion Transformer; denoises latent vectors conditioned on cancer-type labels.

The Diffusion checkpoint takes the VAE as a frozen sub-module so that, at inference, latent encoding / decoding stays consistent with training.

In [ ]:
from scvi.module import VAE
from src.models import GiT
from src.modules import VAEModule, DiffusionModule

vae_backbone = VAE(
    n_input=args.input_size,
    n_batch=args.num_classes,
    n_hidden=args.hidden_size_vae,
    n_latent=args.latent_size,
    n_layers=args.num_layer,
)
git_backbone = GiT(
    latent_size=args.latent_size,
    patch_size=args.patch_size,
    hidden_size=args.hidden_size_sit,
    depth=args.depth,
    num_heads=args.num_heads,
    mlp_ratio=args.mlp_ratio,
    class_dropout_prob=args.class_dropout_prob,
    num_classes=args.num_classes,
    learn_sigma=args.learn_sigma,
)

vae_module = VAEModule.load_from_checkpoint(
    checkpoint_path=VAE_CKPT, args=args, model=vae_backbone, map_location=device,
)
diffusion_module = DiffusionModule.load_from_checkpoint(
    checkpoint_path=DIFF_CKPT, args=args, model=git_backbone, vae=vae_module, map_location=device,
)
diffusion_module.eval().to(device)
print('Loaded VAE + Diffusion modules.')

## 4. Load Spatial Transcriptomics Data

`STDataset` reads the `.h5ad` file and exposes:
- `X` — the raw gene expression matrix
- `get_impute_mask()` — boolean mask over genes; `True` = gene is observed in this platform / panel and should be used as imputation guide; `False` = gene to impute
- `get_names()` — cell / gene name lists
- `get_annotation()` — optional cell-type labels (if present)

In [ ]:
from src.data import STDataset

test_dataset = STDataset('st_test', ST_PATH, args.cancer_type)
test_loader = DataLoader(
    test_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=args.num_cpus,
)

impute_mask = test_dataset.get_impute_mask().to(device)
_, gene_names = test_dataset.get_names()
gene_names = np.array(gene_names)

n_cells = test_dataset.get_num_cell()
n_genes = test_dataset.get_num_gene()
n_observed = int(impute_mask.sum().item())
print(f'Cells: {n_cells} | Total genes: {n_genes} | Observed: {n_observed} | To impute: {n_genes - n_observed}')

## 5. Run Imputation

`DiffusionModule.impute` does:
1. For each batch, encode `x * impute_mask` (observed genes only) through the VAE → guide latent `z_guide`.
2. Forward-diffuse `z_guide` to time `t_forward` (default 0.9).
3. Run reverse ODE/SDE sampling with the GiT denoiser → enhanced latent `z_1`.
4. Decode `z_1` through the VAE → imputed gene expression.
5. Optionally apply a per-gene sparsity floor (set bottom-S% values to 0) using `gene_sparsity_ratio.csv` — this matches the sparsity profile observed in the training reference.

Returns `(X_imputed, Z_latent)` when `return_latent=True`.

In [ ]:
X_imputed, Z_latent = diffusion_module.impute(
    test_loader,
    impute_mask,
    use_sparity_ratio=True,
    return_latent=True,
)
print('X_imputed shape:', X_imputed.shape, '| Z_latent shape:', Z_latent.shape)

## 6. Inspect Results

### 6.1 Marker gene before vs. after
Pick any marker known to be relevant for the cancer type — the demo uses **`EPCAM`** (epithelial marker, common in COAD). Switch `MARKER` to inspect any other gene in the panel.

In [ ]:
MARKER = 'EPCAM'

X_raw = test_dataset.X  # numpy ndarray, shape (n_cells, n_genes)
if MARKER not in gene_names:
    raise KeyError(f'{MARKER} not in gene panel — pick another from gene_names')
g_idx = int(np.where(gene_names == MARKER)[0][0])

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].hist(X_raw[:, g_idx], bins=40, color='#888888')
ax[0].set_title(f'{MARKER}  raw')
ax[0].set_xlabel('expression')
ax[1].hist(X_imputed[:, g_idx], bins=40, color='#C85554')
ax[1].set_title(f'{MARKER}  stPainter')
ax[1].set_xlabel('expression')
plt.tight_layout()
plt.show()

print(f'Non-zero fraction — raw: {(X_raw[:, g_idx] > 0).mean():.2%} | imputed: {(X_imputed[:, g_idx] > 0).mean():.2%}')

### 6.2 UMAP on the enhanced latent

The enhanced latent `Z_latent` is what downstream clustering / cell-type identification should use — it carries more cell-type signal than the raw expression because diffusion has denoised it.

In [ ]:
import anndata as ad

adata = ad.AnnData(X=Z_latent)
labels = test_dataset.get_annotation()
if labels:
    adata.obs['annotation'] = labels
    adata.obs['annotation'] = adata.obs['annotation'].astype('category')

sc.pp.neighbors(adata, use_rep='X', n_neighbors=15)
sc.tl.umap(adata)

color_key = 'annotation' if 'annotation' in adata.obs else None
sc.pl.umap(adata, color=color_key, title=f'stPainter latent (dim={args.latent_size})')

## 7. (Optional) Save Imputed Output

Write the imputed `.h5ad` to disk — equivalent to what the CLI `impute_stPainter.py --output_path ...` produces.
- `layers['imputed']` — sparse matrix of imputed expression
- `obsm['latent']` — enhanced latent embedding

In [ ]:
OUTPUT_PATH = f'./data/imputed_{args.latent_size}/st_COAD_imputed.h5ad'
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

test_dataset.save_imputed(X_imputed)
test_dataset.save_latent(Z_latent)
test_dataset.store_result(OUTPUT_PATH)
print('Wrote', OUTPUT_PATH)

## Next steps

- **Other datasets**: swap `ST_PATH` to any `st_<CANCER>_test.h5ad` you have; update `args.cancer_type` accordingly (the cancer label is one-hot conditioning to the diffusion).
- **Larger latent**: re-run with `LATENT_SIZE = 100` and the matching checkpoints — generally yields better metrics at the cost of inference time.
- **Evaluation**: `test_stPainter.py` runs 5-fold cross-validation gene metrics (PCC / SSIM / RMSE / JS) and clustering metrics (ARI / AMI / Homo / NMI). See README → Getting Started → §3.
- **Train from scratch**: see README → Model Training for the two-stage VAE + Diffusion pipeline.